# LLMOps Regression Gate With Kaggle 2-GPU Evaluation

This notebook practices the multi-GPU LLMOps pattern: split an evaluation set across workers, aggregate metrics with `torch.distributed`, register the candidate run, and apply a promotion regression gate.

## Kaggle Settings

Use the right sidebar:

- Accelerator: `GPU T4 x2` when available.
- Internet: `On`.

The notebook uses Kaggle's preinstalled CUDA PyTorch.

In [ ]:
!nvidia-smi

import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
print('gpu count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

## Clone The Project

In [ ]:
%cd /kaggle/working
!rm -rf llmops-experiment-tracker-and-regression-gate
!git clone https://github.com/msa-1988/llmops-experiment-tracker-and-regression-gate.git
%cd /kaggle/working/llmops-experiment-tracker-and-regression-gate

## Install Dashboard/Reporting Dependencies

Do not reinstall PyTorch on Kaggle. It already has a CUDA-enabled build.

In [ ]:
!pip install -q streamlit pandas matplotlib PyYAML

## Bootstrap Baseline Runs

The regression gate needs a production baseline before it can judge a candidate.

In [ ]:
!python scripts/bootstrap_demo_runs.py
!ls artifacts/runs

## Run 2-GPU Distributed Evaluation

`torchrun` starts two Python workers. Each worker evaluates a different shard of examples on its own GPU. The workers reduce metric sums into one global run record on rank 0.

In [ ]:
!torchrun --standalone --nproc_per_node=2 scripts/distributed_llm_eval.py \
  --baseline baseline-qwen25-base \
  --candidate-run-id candidate-kaggle-ddp-eval \
  --candidate-name 'Qwen2.5-0.5B + Kaggle DDP evaluation' \
  --num-examples 512 \
  --profile optimized \
  --device cuda

## Inspect The Registered Run And Gate Decision

In [ ]:
import json
from pathlib import Path

run = json.loads(Path('artifacts/runs/candidate-kaggle-ddp-eval.json').read_text())
decision = json.loads(Path('artifacts/gate_decisions/candidate-kaggle-ddp-eval_vs_baseline-qwen25-base.json').read_text())

print(json.dumps(run, indent=2))
print(json.dumps(decision, indent=2))

## Export A Promotion Report

In [ ]:
!python scripts/export_run_report.py \
  --baseline baseline-qwen25-base \
  --candidate candidate-kaggle-ddp-eval \
  --output artifacts/kaggle_ddp_run_report.md

print(Path('artifacts/kaggle_ddp_run_report.md').read_text())

## Practice Variations

Change `--profile optimized` to `balanced` or `regressed` and rerun the distributed evaluation cell. You should see how the same gate logic responds to quality, latency, and token-budget changes.